---
title: Environment
execute:
  enabled: true
---

`q.env` reads environment variables and describes the machine where qrt is running. This tutorial uses `demo.env` and `demo-requirements.yml` from the repository root.

## Load and read a `.env` file

`load()` searches upward for a file named `.env`. Pass a path to load a specific file, and use `override=True` when file values should replace variables already present in the process environment.

`values()` parses a specific dotenv file and returns its contents without making any further changes to `os.environ`. The checked-in `demo.env` contains only a safe demonstration token, so its parsed value is shown here. Do not display `values()` output for files containing real credentials.

In [1]:
from pathlib import Path

import qrt as q

demo_env = Path("../../demo.env")
demo_requirements = Path("../../demo-requirements.yml")

q.env.load()  # Search upward for a file named .env
q.env.load(demo_env)  # Load a specific dotenv file
q.env.load(demo_env, override=True)  # Replace existing process values

file_values = q.env.values(demo_env)  # Parse without modifying os.environ
file_values

{'MY_TOKEN': 'ABC123', 'ANOTHER_TOKEN': 'XYZ789'}

## Access loaded variables

After `demo.env` is loaded, `get` returns `MY_TOKEN` as an optional value. The plain-string form of `require` returns the same variable when it must be configured and raises `KeyError` when it is absent.

These cells display the safe token from `demo.env` and then confirm that the required lookup succeeded. Avoid printing real credentials in application code or documentation.

In [2]:
token = q.env.get("MY_TOKEN")
token

'ABC123'

In [3]:
required_token = q.env.require("MY_TOKEN")
"MY_TOKEN is configured" if required_token else "MY_TOKEN is empty"

'MY_TOKEN is configured'

### Missing values

`get` returns a fallback for optional configuration. `require` raises `KeyError` when a required variable is absent. The next cell catches only to keep the tutorial executable; the displayed object is the real exception raised by `q.env`.

In [4]:
optional_value = q.env.get("MISSING_SETTING", "fallback")
missing_error = None

try:
    q.env.require("MISSING_SETTING")
except KeyError as error:
    missing_error = error

optional_value, missing_error

('fallback',
 KeyError('Required environment variable is not set: MISSING_SETTING'))

## Inspect the runtime

The other half of `q.env` describes the current machine: operating system, filesystem capacity, usable PyTorch accelerators, and preferred device. `info` combines those details into an immutable, JSON-friendly snapshot.

In [5]:
environment = q.env.info()
environment.as_dict()

{'operating_system': {'name': 'Linux',
  'release': '7.0.0-27-generic',
  'version': '#27-Ubuntu SMP PREEMPT_DYNAMIC Thu Jun 18 19:13:49 UTC 2026',
  'machine': 'x86_64'},
 'disk': {'path': '/home/hi/qrt/docs/env',
  'total': 2011838955520,
  'used': 127739265024,
  'free': 1781828300800},
 'accelerators': (),
 'device': 'cpu',
 'cpu_count': 24}

### Environment report

`report` presents the same snapshot as a compact Rich table and returns it for programmatic use.

In [6]:
_ = q.env.report()

                            qrt environment                            
┌──────────────────┬──────────────────────────────────────────────────┐
│ OS               │ Linux 7.0.0-27-generic (x86_64)                  │
│ CPU cores        │ 24                                               │
│ Disk             │ 1.6 TiB free of 1.8 TiB at /home/hi/qrt/docs/env │
│ Accelerator      │ None detected                                    │
│ Preferred device │ cpu                                              │
└──────────────────┴──────────────────────────────────────────────────┘

## Require a runtime environment

`require` behaves according to its input:

| Input | Behavior |
|---|---|
| A plain string such as `"MY_TOKEN"` | Return that required environment variable |
| A `Path`, or a string ending in `.yml`/`.yaml` | Validate the machine against that requirements file |

The checked-in requirements file contains:

```yaml
disk:
  path: .
  free: 1 B

env:
  - MY_TOKEN
```

The disk path is resolved relative to the YAML file. Validation succeeds because `demo.env` has already loaded `MY_TOKEN`; unmet conditions would be collected in `EnvironmentRequirementError.failures`.

In [7]:
validated = q.env.require(demo_requirements)
{
    "requirements_met": True,
    "device": validated.device,
    "disk": str(validated.disk.path),
}

{'requirements_met': True, 'device': 'cpu', 'disk': '/home/hi/qrt'}